# Rare-term weighting on Kaggle — baseline vs. hapax_binary vs. inverse_sqrt_freq

Adapted from `scripts/kaggle_deberta_grid.ipynb` (same environment pattern: data cloned via
git, not uploaded; pinned installs; GPU T4 x2). This notebook does **not** touch
`origin/main` — the rare-term-weighting change lives only on the local
`feature/rare-term-weighting` branch and is never pushed. It is applied here as a patch,
written into a fresh clone of `origin/main`, so nothing about this run depends on that
branch existing anywhere but on Khaled's own machine.

**Two things to set in the right-hand sidebar before running:**
1. **Session options → Accelerator → GPU T4 x2** (or P100)
2. **Session options → Internet → On** (needs a phone-verified account) — the clone and the
   HuggingFace download both need it.

Model/LR/epochs default to **deberta-v3-base, 1e-5, 5 epochs — T10's selected config**
(see cell 6). Swap to `bert-base-cased` / `3e-5` there if a faster, non-final run is what's
wanted instead.

In [ ]:
# 1. Kaggle guard, internet, GPU, and the command helper.
import os, sys, socket, subprocess, pathlib

if not pathlib.Path('/kaggle').is_dir():
    raise SystemExit('This notebook is for Kaggle. On Colab use colab_deberta_grid.ipynb.')

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError as e:
    raise SystemExit(
        f'No internet ({e}). Kaggle disables it by default.\n'
        'Sidebar -> Session options -> Internet -> On (needs a phone-verified account).')

import torch
assert torch.cuda.is_available(), 'No GPU. Sidebar -> Session options -> Accelerator -> GPU.'
print(torch.cuda.get_device_name(0), '|', torch.__version__, '| cuda', torch.version.cuda)

WORK = pathlib.Path('/kaggle/working')        # persisted as notebook Output
REPO = pathlib.Path('/tmp/ate-acter')         # scratch: repo + corpus stay out of Output

def run(*args, cwd=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         cwd=cwd, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(str(a) for a in args)}')


In [ ]:
# 2. Clone the project (origin/main -- no local branch, no rare-term-weighting code
#    yet) and the corpus into /tmp.
import shutil
shutil.rmtree(REPO, ignore_errors=True)
run('git', 'clone', '-q', 'https://github.com/ahmedwaleedaref/ATE-ACTER.git', REPO)
run('git', 'clone', '-q', 'https://github.com/AylaRT/ACTER.git', REPO / 'data/raw/ACTER')
run('git', 'checkout', '-q', 'f05b09e985cad37eeaa8daa8b3f383197aa5324e',
    cwd=REPO / 'data/raw/ACTER')
assert (REPO / 'data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                     capture_output=True, text=True).stdout)


In [ ]:
# 3. Apply the rare-term-weighting change. This is the local, unpushed
#    feature/rare-term-weighting branch's diff against the exact commit cell 2 just
#    cloned -- written here instead of pulled from GitHub, since it is not there.
#    Base64, not a triple-quoted string: the patch and the new files contain their
#    own triple-quoted docstrings, which would otherwise close the string early.
import base64

PATCH_B64 = "ZGlmZiAtLWdpdCBhL2NvbmZpZ3MvdHJhaW4uanNvbiBiL2NvbmZpZ3MvdHJhaW4uanNvbgppbmRleCBiMjdiYWEwLi4zY2QwOGQ1IDEwMDY0NAotLS0gYS9jb25maWdzL3RyYWluLmpzb24KKysrIGIvY29uZmlncy90cmFpbi5qc29uCkBAIC0zNCw1ICszNCwxMiBAQAogICAgICJkb21haW5zIjogWyJ3aW5kIl0sCiAgICAgInNwbGl0cyI6IFsidHJhaW4iXQogICB9LAotICAiX2ZpbHRlcl9ub3RlIjogImRhdGFfbGF5b3V0Lm1kIHNlY3Rpb24gMi4gQXBwbGllZCBieSB0aGUgd3JhcHBlciBhZnRlciBsb2FkX2RvbWFpbiByZXR1cm5zLCBuZXZlciBpbnNpZGUgdGhlIGxvYWRlciwgYW5kIHBhc3NlZCB0byBidWlsZF9leGFtcGxlcyBhcyBhbiBleHBsaWNpdCBwYXJhbWV0ZXI6IHNldCBkb21haW5zIHRvIFtdIHRvIHJ1biB3aXRoIGl0IG9mZiB3aXRob3V0IGVkaXRpbmcgY29kZS4gSXQgbXVzdCBuZXZlciBhcHBseSB0byBlcXVpIG9yIGh0ZmwuIgotfQorICAiX2ZpbHRlcl9ub3RlIjogImRhdGFfbGF5b3V0Lm1kIHNlY3Rpb24gMi4gQXBwbGllZCBieSB0aGUgd3JhcHBlciBhZnRlciBsb2FkX2RvbWFpbiByZXR1cm5zLCBuZXZlciBpbnNpZGUgdGhlIGxvYWRlciwgYW5kIHBhc3NlZCB0byBidWlsZF9leGFtcGxlcyBhcyBhbiBleHBsaWNpdCBwYXJhbWV0ZXI6IHNldCBkb21haW5zIHRvIFtdIHRvIHJ1biB3aXRoIGl0IG9mZiB3aXRob3V0IGVkaXRpbmcgY29kZS4gSXQgbXVzdCBuZXZlciBhcHBseSB0byBlcXVpIG9yIGh0ZmwuIiwKKworICAicmFyZV90ZXJtX3dlaWdodGluZyI6IHsKKyAgICAiZW5hYmxlZCI6IGZhbHNlLAorICAgICJmb3JtdWxhIjogImludmVyc2Vfc3FydF9mcmVxIiwKKyAgICAiaGFwYXhfd2VpZ2h0IjogMi4wCisgIH0sCisgICJfcmFyZV90ZXJtX25vdGUiOiAic3JjL2RhdGEvcmFyZV90ZXJtcy5weS4gRnJlcXVlbmN5IGlzIGNvdW50ZWQgZnJvbSB0cmFpbl9kb21haW5zIG9ubHkgKGNvbmZpZ3MvZGF0YS5qc29uKSwgcG9zdCBzaG9ydF9zZW50ZW5jZV9maWx0ZXIgLS0gbmV2ZXIgZXF1aSBvciBodGZsLiBmb3JtdWxhIGlzICdoYXBheF9iaW5hcnknIChoYXBheF93ZWlnaHQgZm9yIGZyZXE9PTEsIGVsc2UgMS4wKSBvciAnaW52ZXJzZV9zcXJ0X2ZyZXEnICgxL3NxcnQoZnJlcSkpOyBoYXBheF93ZWlnaHQgaXMgcmVhZCBvbmx5IGJ5IGhhcGF4X2JpbmFyeS4gZW5hYmxlZD1mYWxzZSBpcyBhIG5vLW9wOiBldmVyeSBzdWJ3b3JkIHdlaWdodCBpcyAxLjAsIG1hdGNoaW5nIHByZS1yYXJlLXRlcm0td2VpZ2h0aW5nIGJlaGF2aW91ciBleGFjdGx5LiIKK30KXCBObyBuZXdsaW5lIGF0IGVuZCBvZiBmaWxlCmRpZmYgLS1naXQgYS9zcmMvZGF0YS9kYXRhc2V0LnB5IGIvc3JjL2RhdGEvZGF0YXNldC5weQppbmRleCBiNDAxNWY2Li40ZDhjZDU5IDEwMDY0NAotLS0gYS9zcmMvZGF0YS9kYXRhc2V0LnB5CisrKyBiL3NyYy9kYXRhL2RhdGFzZXQucHkKQEAgLTMyLDYgKzMyLDEyIEBAIGZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlciwgRGF0YXNldCwgU2FtcGxlcgogZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Ub2tlbml6ZXIsIERhdGFDb2xsYXRvckZvclRva2VuQ2xhc3NpZmljYXRpb24KIAogZnJvbSBzcmMuZGF0YS5hbGlnbiBpbXBvcnQgSUQyTEFCRUwsIElHTk9SRV9JTkRFWCwgTEFCRUwySUQsIGFsaWduX2xhYmVscworZnJvbSBzcmMuZGF0YS5yYXJlX3Rlcm1zIGltcG9ydCAoCisgICAgQkFTRUxJTkVfV0VJR0hULAorICAgIGFsaWduX3dlaWdodHMsCisgICAgY291bnRfdGVybV9mcmVxdWVuY2llcywKKyAgICB0b2tlbl93ZWlnaHRzX2Zvcl9zZW50ZW5jZSwKKykKIGZyb20gc3JjLnN0YXRzLmxvYWRpbmcgaW1wb3J0IERhdGFDb25maWcsIGxvYWRfY29uZmlnLCBsb2FkX2RvbWFpbgogCiBfUkVQT19ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KQEAgLTcxLDYgKzc3LDkgQEAgY2xhc3MgVHJhaW5Db25maWc6CiAgICAgZmlsdGVyX21heF90b2tlbnM6IGludAogICAgIGZpbHRlcl9kb21haW5zOiB0dXBsZVtzdHIsIC4uLl0KICAgICBmaWx0ZXJfc3BsaXRzOiB0dXBsZVtzdHIsIC4uLl0KKyAgICByYXJlX3Rlcm1fd2VpZ2h0aW5nX2VuYWJsZWQ6IGJvb2wKKyAgICByYXJlX3Rlcm1fZm9ybXVsYTogc3RyICAgICAgICAgICMgImhhcGF4X2JpbmFyeSIgfCAiaW52ZXJzZV9zcXJ0X2ZyZXEiCisgICAgcmFyZV90ZXJtX2hhcGF4X3dlaWdodDogZmxvYXQgICAjIG9ubHkgcmVhZCBieSAiaGFwYXhfYmluYXJ5IgogICAgIHJhdzogZGljdCAgICAgICAgICAgIyB0aGUgZmlsZSBhcy1sb2FkZWQsIGZvciB0aGUgcGVyLXJ1biByZXN1bHRzLyBoZWFkZXIKIAogICAgIGRlZiBmaWx0ZXJfZm9yKHNlbGYsIGRvbWFpbjogc3RyLCBzcGxpdDogc3RyKSAtPiBpbnQgfCBOb25lOgpAQCAtMTAzLDYgKzExMiwxMCBAQCBkZWYgbG9hZF90cmFpbl9jb25maWcocGF0aD1Ob25lKSAtPiBUcmFpbkNvbmZpZzoKICAgICApCiAKICAgICBmaWx0ID0gcmF3WyJzaG9ydF9zZW50ZW5jZV9maWx0ZXIiXQorICAgICMgT3B0aW9uYWwgYmxvY2s6IGFic2VudCBlbnRpcmVseSBpbiBvbGRlciBjb25maWdzLCB3aGljaCBtdXN0IGtlZXAKKyAgICAjIGxvYWRpbmcgdW5jaGFuZ2VkIC0tIHRoZSBmZWF0dXJlIGRlZmF1bHRzIE9GRiwgbWF0Y2hpbmcgY3VycmVudAorICAgICMgYmVoYXZpb3VyIGV4YWN0bHkgKGJ1aWxkX2V4YW1wbGVzKCkgZmFsbHMgYmFjayB0byB3ZWlnaHQgMS4wIGV2ZXJ5d2hlcmUpLgorICAgIHJhcmUgPSByYXcuZ2V0KCJyYXJlX3Rlcm1fd2VpZ2h0aW5nIiwge30pCiAgICAgcmV0dXJuIFRyYWluQ29uZmlnKAogICAgICAgICBtb2RlbF9uYW1lPXJhd1sibW9kZWxfbmFtZSJdLAogICAgICAgICBoZl9jYWNoZV9kaXI9cmF3WyJoZl9jYWNoZV9kaXIiXSwKQEAgLTEyNiw2ICsxMzksOSBAQCBkZWYgbG9hZF90cmFpbl9jb25maWcocGF0aD1Ob25lKSAtPiBUcmFpbkNvbmZpZzoKICAgICAgICAgZmlsdGVyX21heF90b2tlbnM9ZmlsdFsibWF4X3Rva2VucyJdLAogICAgICAgICBmaWx0ZXJfZG9tYWlucz10dXBsZShmaWx0WyJkb21haW5zIl0pLAogICAgICAgICBmaWx0ZXJfc3BsaXRzPXR1cGxlKGZpbHRbInNwbGl0cyJdKSwKKyAgICAgICAgcmFyZV90ZXJtX3dlaWdodGluZ19lbmFibGVkPXJhcmUuZ2V0KCJlbmFibGVkIiwgRmFsc2UpLAorICAgICAgICByYXJlX3Rlcm1fZm9ybXVsYT1yYXJlLmdldCgiZm9ybXVsYSIsICJpbnZlcnNlX3NxcnRfZnJlcSIpLAorICAgICAgICByYXJlX3Rlcm1faGFwYXhfd2VpZ2h0PWZsb2F0KHJhcmUuZ2V0KCJoYXBheF93ZWlnaHQiLCAyLjApKSwKICAgICAgICAgcmF3PXJhdywKICAgICApCiAKQEAgLTE2Niw2ICsxODIsMTAgQEAgY2xhc3MgRXhhbXBsZToKICAgICBhdHRlbnRpb25fbWFzazogbGlzdFtpbnRdCiAgICAgbGFiZWxzOiBsaXN0W2ludF0KICAgICB3b3JkX2lkczogbGlzdFtpbnQgfCBOb25lXQorICAgICMgUmFyZS10ZXJtIGxvc3Mgd2VpZ2h0IHBlciBzdWJ3b3JkIHBvc2l0aW9uLCBzYW1lIHNoYXBlIGFzIGBgbGFiZWxzYGAuCisgICAgIyBBbGwgMS4wIChCQVNFTElORV9XRUlHSFQpIHdoZW4gcmFyZS10ZXJtIHdlaWdodGluZyBpcyBvZmYgb3IgdGhpcyBzcGxpdAorICAgICMgaXMgbm90IHRyYWluIC0tIGEgcHVyZSBuby1vcCBpbiB0aGF0IGNhc2UsIG5ldmVyIHJlYWQgYnkgZXZhbHVhdGUoKS4KKyAgICBzdWJ3b3JkX3dlaWdodHM6IGxpc3RbZmxvYXRdCiAKICAgICBAcHJvcGVydHkKICAgICBkZWYgbl9zdWJ3b3JkcyhzZWxmKSAtPiBpbnQ6CkBAIC0yMjcsNiArMjQ3LDkgQEAgZGVmIGJ1aWxkX2V4YW1wbGVzKAogICAgIG1heF9sZW5ndGg6IGludCB8IE5vbmUsCiAgICAgZmlsdGVyX21heF90b2tlbnM6IGludCB8IE5vbmUsCiAgICAgZGF0YV9jZmc6IERhdGFDb25maWcgfCBOb25lID0gTm9uZSwKKyAgICB0ZXJtX2ZyZXF1ZW5jaWVzOiBkaWN0W3N0ciwgaW50XSB8IE5vbmUgPSBOb25lLAorICAgIHJhcmVfdGVybV9mb3JtdWxhOiBzdHIgPSAiaW52ZXJzZV9zcXJ0X2ZyZXEiLAorICAgIHJhcmVfdGVybV9oYXBheF93ZWlnaHQ6IGZsb2F0ID0gMi4wLAogKSAtPiBsaXN0W0V4YW1wbGVdOgogICAgICIiIlRva2VuaXplIGFuZCBhbGlnbiBvbmUgZG9tYWluLgogCkBAIC0yMzksNiArMjYyLDEzIEBAIGRlZiBidWlsZF9leGFtcGxlcygKICAgICB0aGUgbG9hZGVyJ3MgbGFiZWxzIGZvciBleGFjdCBlcXVhbGl0eS4gVW5kZXIgdHJ1bmNhdGlvbiB0aGUgY29tcGFyaXNvbgogICAgIGZhaWxzIG9uIHRoZSA4IGNvcnB1cy13aWRlIHNlbnRlbmNlcyB0aGF0IGV4Y2VlZCAyNTYgcGllY2VzIGZvciBhIHJlYXNvbgogICAgIHRoYXQgaXMgbm90IGEgYnVnIC0tIGBgZGF0YV9sYXlvdXQubWRgYCBzZWN0aW9uIDguMi4KKworICAgIGBgdGVybV9mcmVxdWVuY2llc2BgIGlzIGBgTm9uZWBgIGJ5IGRlZmF1bHQ6IGV2ZXJ5IGBgc3Vid29yZF93ZWlnaHRzYGAKKyAgICBlbnRyeSBpcyB0aGVuIGBgQkFTRUxJTkVfV0VJR0hUYGAgKDEuMCksIGEgcHVyZSBuby1vcCAtLSB0aGlzIGlzIHdoYXQgZGV2CisgICAgYW5kIHRlc3QgY2FsbGVycywgYW5kIGFueSBjYWxsZXIgZnJvbSBiZWZvcmUgcmFyZS10ZXJtIHdlaWdodGluZyBleGlzdGVkLAorICAgIGdldCBhdXRvbWF0aWNhbGx5LiBQYXNzIGEgdGFibGUgKGJ1aWx0IGJ5CisgICAgYGBzcmMuZGF0YS5yYXJlX3Rlcm1zLmNvdW50X3Rlcm1fZnJlcXVlbmNpZXNgYCBvdmVyIHRoZSB0cmFpbiBkb21haW5zCisgICAgb25seSkgdG8gdHVybiBwZXItdGVybSB3ZWlnaHRpbmcgb24gZm9yIHRoaXMgY2FsbC4KICAgICAiIiIKICAgICBhc3NlcnQgdHJ1bmNhdGlvbiBvciBtYXhfbGVuZ3RoIGlzIE5vbmUsICJtYXhfbGVuZ3RoIGlzIG1lYW5pbmdsZXNzIHdpdGggdHJ1bmNhdGlvbiBvZmYiCiAgICAgZG9jdW1lbnRzID0gbG9hZF9kb21haW4oZG9tYWluLCBkYXRhX2NmZykKQEAgLTI2MCw2ICsyOTAsMTUgQEAgZGVmIGJ1aWxkX2V4YW1wbGVzKAogICAgICAgICAgICAgICAgIG1heF9sZW5ndGg9bWF4X2xlbmd0aCwKICAgICAgICAgICAgICkKICAgICAgICAgICAgIHdvcmRfaWRzID0gZW5jb2Rpbmcud29yZF9pZHMoKQorCisgICAgICAgICAgICBpZiB0ZXJtX2ZyZXF1ZW5jaWVzIGlzIG5vdCBOb25lOgorICAgICAgICAgICAgICAgIHRva2VuX3dlaWdodHMgPSB0b2tlbl93ZWlnaHRzX2Zvcl9zZW50ZW5jZSgKKyAgICAgICAgICAgICAgICAgICAgdG9rZW5zLCBsYWJlbHMsIHRlcm1fZnJlcXVlbmNpZXMsCisgICAgICAgICAgICAgICAgICAgIGZvcm11bGE9cmFyZV90ZXJtX2Zvcm11bGEsIGhhcGF4X3dlaWdodD1yYXJlX3Rlcm1faGFwYXhfd2VpZ2h0KQorICAgICAgICAgICAgICAgIHN1YndvcmRfd2VpZ2h0cyA9IGFsaWduX3dlaWdodHMod29yZF9pZHMsIHRva2VuX3dlaWdodHMpCisgICAgICAgICAgICBlbHNlOgorICAgICAgICAgICAgICAgIHN1YndvcmRfd2VpZ2h0cyA9IFtCQVNFTElORV9XRUlHSFRdICogbGVuKHdvcmRfaWRzKQorCiAgICAgICAgICAgICBleGFtcGxlcy5hcHBlbmQoRXhhbXBsZSgKICAgICAgICAgICAgICAgICBkb21haW49ZG9tYWluLAogICAgICAgICAgICAgICAgIGZpbGVfaWQ9ZG9jLmZpbGVfaWQsCkBAIC0yNzIsNiArMzExLDcgQEAgZGVmIGJ1aWxkX2V4YW1wbGVzKAogICAgICAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPWVuY29kaW5nWyJhdHRlbnRpb25fbWFzayJdLAogICAgICAgICAgICAgICAgIGxhYmVscz1hbGlnbl9sYWJlbHMod29yZF9pZHMsIGxhYmVscywgTEFCRUwySUQpLAogICAgICAgICAgICAgICAgIHdvcmRfaWRzPXdvcmRfaWRzLAorICAgICAgICAgICAgICAgIHN1YndvcmRfd2VpZ2h0cz1zdWJ3b3JkX3dlaWdodHMsCiAgICAgICAgICAgICApKQogICAgIHJldHVybiBleGFtcGxlcwogCkBAIC0yOTUsNiArMzM1LDcgQEAgY2xhc3MgQVRFRGF0YXNldChEYXRhc2V0KToKICAgICAgICAgICAgICJpbnB1dF9pZHMiOiBleGFtcGxlLmlucHV0X2lkcywKICAgICAgICAgICAgICJhdHRlbnRpb25fbWFzayI6IGV4YW1wbGUuYXR0ZW50aW9uX21hc2ssCiAgICAgICAgICAgICAibGFiZWxzIjogZXhhbXBsZS5sYWJlbHMsCisgICAgICAgICAgICAid2VpZ2h0cyI6IGV4YW1wbGUuc3Vid29yZF93ZWlnaHRzLCAgIyBhbGwgMS4wIHdoZW4gcmFyZS10ZXJtIHdlaWdodGluZyBpcyBvZmYKICAgICAgICAgICAgICJleGFtcGxlX2luZGV4IjogaW5kZXgsI3RoaXMgdXNlZCBpZiB3ZSB3YW50IHRvIGxvb2sgdXAgZm9yIHNvbWV0aGluZyBlbHNlIHVzZSB0aGlzIGluZGV4IC4gCiAgICAgICAgIH0KICAgICAjdGhpcyBpcyBhIG11c3Qgd2UgbmVlZCB0byBrbm93IHRoZSBtYXgtdG9rZW4tc2l6ZSBvZiBleGFtcGxlIGluIGJhdGNoIGJjcyB3ZSB3aWxsIGRvIGR5bmFtaWMgcGFkZGluZyAKQEAgLTMyMSwxMCArMzYyLDI3IEBAIGNsYXNzIENvbGxhdG9yOgogICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBmZWF0dXJlczogbGlzdFtkaWN0XSkgLT4gZGljdDoKICAgICAgICAgI3JlbW92ZSB0aGUgZXhhbXBsZV9pbmRleCB5b3UgY2Fubm90IGlucHV0IGl0IHRvIGNvbGxhdG9yIAogICAgICAgICBpbmRpY2VzID0gW2ZlYXR1cmUucG9wKCJleGFtcGxlX2luZGV4IikgZm9yIGZlYXR1cmUgaW4gZmVhdHVyZXNdIAorICAgICAgICAjICJ3ZWlnaHRzIiBpcyBub3QgYSBmaWVsZCBEYXRhQ29sbGF0b3JGb3JUb2tlbkNsYXNzaWZpY2F0aW9uIGtub3dzIGhvdworICAgICAgICAjIHRvIHBhZCAoaXQgb25seSBzcGVjaWFsLWNhc2VzICJsYWJlbHMiIGFuZCB0aGUgdG9rZW5pemVyJ3Mgb3duIG1vZGVsCisgICAgICAgICMgaW5wdXRzKSAtLSBwb3AgaXQgb3V0IHRoZSBzYW1lIHdheSwgcGFkIGl0IGJ5IGhhbmQgYWZ0ZXJ3YXJkcy4KKyAgICAgICAgd2VpZ2h0cyA9IFtmZWF0dXJlLnBvcCgid2VpZ2h0cyIpIGZvciBmZWF0dXJlIGluIGZlYXR1cmVzXQogICAgICAgICAjbm93IGlucHV0IGlzIGp1c3QgYSBbeyJpbnB1dF9pZHMiOiBbLi4uXSwgImF0dGVudGlvbl9tYXNrIjogWy4uLl0sICJsYWJlbHMiOiBbLi4uXX0sICJpbnB1dF9pZHMiOiBbLi4uXSwgImF0dGVudGlvbl9tYXNrIjogWy4uLl0sICJsYWJlbHMiOiBbLi4uXSwgImV4YW1wbGVfaW5kZXgiOiAxMiBdCiAgICAgICAgICN0aGUgY29sbGF0b3Igd2lsbCBmaW5kIGxvbmdlc3Qgc2VudGVuY2UgYW5kIGRvIHBhZGRpbmcgZGVwZW5kaW5nIG9uIGl0IGFuZCBzdGFjayBlYWNoIGZpZWxkIGludG8gYSB0ZW5zb3IgeyJpbnB1dF9pZHMiOiB0ZW5zb3IoQiwgTCksICJhdHRlbnRpb25fbWFzayI6IHRlbnNvcihCLCBMKSwgImxhYmVscyI6IHRlbnNvcihCLCBMKX0KICAgICAgICAgI0IgaXMgbnVtYmVyIG9mIGV4YW1wbGUgaW4gYmF0Y2ggaWYgeW91IGFjY2VzcyB0aGUgZmlyc3QgZXhhbXBsZSB5b3Ugd2lsbCBmaW5kIDEgdmVjdG9yIHdpdGggc2l6ZSBMIG51bWJlciBvZiBzdWJ3b3JkIHBvc2l0aW9ucyBpbiB0aGUgbG9uZ2VzdCBzZXF1ZW5jZSBpbiB0aGlzIGJhdGNoIHRoZSA3NjggZG9lcyBub3QgZ2V0IGhlcmUgeWV0IC4KICAgICAgICAgYmF0Y2ggPSBzZWxmLmNvbGxhdG9yKGZlYXR1cmVzKQorCisgICAgICAgICMgUGFkICJ3ZWlnaHRzIiB0byB0aGUgZXhhY3Qgc2FtZSAoYmF0Y2hfbWF4X2xlbmd0aCwgc2lkZSkgdGhlCisgICAgICAgICMgY29sbGF0b3IganVzdCB1c2VkIGZvciAibGFiZWxzIiAtLSBJR05PUkVfV0VJR0hUICgwLjApIGlzIHRoZSBmbG9hdAorICAgICAgICAjIGFuYWxvZ3VlIG9mIGxhYmVsX3BhZF90b2tlbl9pZCAoLTEwMCk6IGEgc2VudGluZWwgZm9yICJubyByZWFsCisgICAgICAgICMgcG9zaXRpb24gaGVyZSIsIG5ldmVyIGl0c2VsZiBsb2FkLWJlYXJpbmcgZm9yIHRoZSBsb3NzIG1hc2suCisgICAgICAgIG1heF9sZW4gPSBiYXRjaFsibGFiZWxzIl0uc2hhcGVbMV0KKyAgICAgICAgcGFkZGluZ19zaWRlID0gc2VsZi5jb2xsYXRvci50b2tlbml6ZXIucGFkZGluZ19zaWRlCisgICAgICAgIHBhZGRlZF93ZWlnaHRzID0gW10KKyAgICAgICAgZm9yIHcgaW4gd2VpZ2h0czoKKyAgICAgICAgICAgIHBhZCA9IFswLjBdICogKG1heF9sZW4gLSBsZW4odykpCisgICAgICAgICAgICBwYWRkZWRfd2VpZ2h0cy5hcHBlbmQocGFkICsgdyBpZiBwYWRkaW5nX3NpZGUgPT0gImxlZnQiIGVsc2UgdyArIHBhZCkKKyAgICAgICAgYmF0Y2hbIndlaWdodHMiXSA9IHRvcmNoLnRlbnNvcihwYWRkZWRfd2VpZ2h0cywgZHR5cGU9dG9yY2guZmxvYXQpCisKICAgICAgICAgI2xldHMgcHV0IGluZGljZXMgYmFjayBvbmUgaW5kZXggcGVyIGV4YW1wbGUgdGhvc2UgaW5kaWNlcyBhcmUgbmVjc3NhcnkgYmNzIHdlIG5lZWQgdG8gZ28gYmFjayB0byAoZmlsZV9pZCwgc2VudF9pZHgpCiAgICAgICAgIGJhdGNoWyJleGFtcGxlX2luZGV4Il0gPSB0b3JjaC50ZW5zb3IoaW5kaWNlcywgZHR5cGU9dG9yY2gubG9uZykKICAgICAgICAgcmV0dXJuIGJhdGNoCkBAIC00MzUsNiArNDkzLDE3IEBAIGRlZiBidWlsZF9zcGxpdHModHJhaW5fY2ZnOiBUcmFpbkNvbmZpZywgZGF0YV9jZmc6IERhdGFDb25maWcgfCBOb25lID0gTm9uZSwKICAgICAgICAgInRlc3QiOiAoZGF0YV9jZmcudGVzdF9kb21haW4sKSwKICAgICB9CiAKKyAgICAjIFJhcmUtdGVybSB3ZWlnaHRpbmc6IGZyZXF1ZW5jeSBpcyBjb3VudGVkIG9uY2UsIGZyb20gdGhlIHRyYWluIGRvbWFpbnMKKyAgICAjIG9ubHksIGJlZm9yZSBhbnkgRXhhbXBsZSBpcyBidWlsdCAtLSBuZXZlciBmcm9tIGRldi90ZXN0IChlcXVpL2h0ZmwpLAorICAgICMgYW5kIG5ldmVyIHBlci1zZW50ZW5jZSAoYSB0ZXJtJ3Mgd2VpZ2h0IGlzIGNvbnN0YW50IGFjcm9zcyB0aGUgc3BsaXQpLgorICAgIHRlcm1fZnJlcXVlbmNpZXM6IGRpY3Rbc3RyLCBpbnRdIHwgTm9uZSA9IE5vbmUKKyAgICBpZiB0cmFpbl9jZmcucmFyZV90ZXJtX3dlaWdodGluZ19lbmFibGVkOgorICAgICAgICB0ZXJtX2ZyZXF1ZW5jaWVzID0gY291bnRfdGVybV9mcmVxdWVuY2llcygKKyAgICAgICAgICAgIGRvbWFpbnNbInRyYWluIl0sCisgICAgICAgICAgICBmaWx0ZXJfbWF4X3Rva2Vucz17ZDogdHJhaW5fY2ZnLmZpbHRlcl9mb3IoZCwgInRyYWluIikgZm9yIGQgaW4gZG9tYWluc1sidHJhaW4iXX0sCisgICAgICAgICAgICBkYXRhX2NmZz1kYXRhX2NmZywKKyAgICAgICAgKQorCiAgICAgZGF0YXNldHM6IGRpY3Rbc3RyLCBBVEVEYXRhc2V0XSA9IHt9CiAgICAgbG9hZGVyczogZGljdFtzdHIsIERhdGFMb2FkZXJdID0ge30KICAgICBmb3Igc3BsaXQsIHNwbGl0X2RvbWFpbnMgaW4gZG9tYWlucy5pdGVtcygpOgpAQCAtNDQ3LDYgKzUxNiwxMiBAQCBkZWYgYnVpbGRfc3BsaXRzKHRyYWluX2NmZzogVHJhaW5Db25maWcsIGRhdGFfY2ZnOiBEYXRhQ29uZmlnIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICAgICAgbWF4X2xlbmd0aD10cmFpbl9jZmcubWF4X2xlbmd0aCBpZiB0cmFpbl9jZmcudHJ1bmNhdGlvbiBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgZmlsdGVyX21heF90b2tlbnM9dHJhaW5fY2ZnLmZpbHRlcl9mb3IoZG9tYWluLCBzcGxpdCksCiAgICAgICAgICAgICAgICAgZGF0YV9jZmc9ZGF0YV9jZmcsCisgICAgICAgICAgICAgICAgIyBkZXYvdGVzdCBleGFtcGxlcyBhcmUgbmV2ZXIgZmVkIHRvIHRyYWluX3N0ZXAsIHNvIHRoZXkgZ2V0CisgICAgICAgICAgICAgICAgIyBubyB0ZXJtX2ZyZXF1ZW5jaWVzIC0tIGJ1aWxkX2V4YW1wbGVzKCkgdGhlbiBsZWF2ZXMgZXZlcnkKKyAgICAgICAgICAgICAgICAjIHdlaWdodCBhdCBCQVNFTElORV9XRUlHSFQsIGEgcHVyZSBuby1vcC4KKyAgICAgICAgICAgICAgICB0ZXJtX2ZyZXF1ZW5jaWVzPXRlcm1fZnJlcXVlbmNpZXMgaWYgc3BsaXQgPT0gInRyYWluIiBlbHNlIE5vbmUsCisgICAgICAgICAgICAgICAgcmFyZV90ZXJtX2Zvcm11bGE9dHJhaW5fY2ZnLnJhcmVfdGVybV9mb3JtdWxhLAorICAgICAgICAgICAgICAgIHJhcmVfdGVybV9oYXBheF93ZWlnaHQ9dHJhaW5fY2ZnLnJhcmVfdGVybV9oYXBheF93ZWlnaHQsCiAgICAgICAgICAgICApKQogICAgICAgICBkYXRhc2V0ID0gQVRFRGF0YXNldChleGFtcGxlcykKICAgICAgICAgZGF0YXNldHNbc3BsaXRdID0gZGF0YXNldApAQCAtNDYwLDQgKzUzNSw0IEBAIGRlZiBidWlsZF9zcGxpdHModHJhaW5fY2ZnOiBUcmFpbkNvbmZpZywgZGF0YV9jZmc6IERhdGFDb25maWcgfCBOb25lID0gTm9uZSwKICAgICAgICAgICAgIHNlZWQ9c2VlZCBpZiBzcGxpdCA9PSAidHJhaW4iIGVsc2UgTm9uZSwKICAgICAgICAgKQogCi0gICAgcmV0dXJuIFNwbGl0cyhsb2FkZXJzPWxvYWRlcnMsIGRhdGFzZXRzPWRhdGFzZXRzLCBkb21haW5zPWRvbWFpbnMpCisgICAgcmV0dXJuIFNwbGl0cyhsb2FkZXJzPWxvYWRlcnMsIGRhdGFzZXRzPWRhdGFzZXRzLCBkb21haW5zPWRvbWFpbnMpClwgTm8gbmV3bGluZSBhdCBlbmQgb2YgZmlsZQpkaWZmIC0tZ2l0IGEvc3JjL21vZGVscy9ydW5fdHJhaW4ucHkgYi9zcmMvbW9kZWxzL3J1bl90cmFpbi5weQppbmRleCBhMTcxYTUzLi5iMzRkNDI4IDEwMDY0NAotLS0gYS9zcmMvbW9kZWxzL3J1bl90cmFpbi5weQorKysgYi9zcmMvbW9kZWxzL3J1bl90cmFpbi5weQpAQCAtMTg2LDYgKzE4NiwxOCBAQCBkZWYgbWFpbigpIC0+IE5vbmU6CiAgICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ncm91cCIsIGRlZmF1bHQ9IiIsCiAgICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJzdWJkaXJlY3RvcnkgdW5kZXIgcmVzdWx0cy9ydW5zLyB0byB3cml0ZSBpbnRvLiBUOSBnaXZlcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJlYWNoIGdyaWQgY2VsbCBpdHMgb3duLCBzbyBjZWxscyBjYW5ub3Qgb3ZlcndyaXRlIGVhY2ggb3RoZXIiKQorICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmFyZS10ZXJtLXdlaWdodGluZyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCisgICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJ0dXJuIG9uIHNyYy9kYXRhL3JhcmVfdGVybXMucHkgd2VpZ2h0aW5nIGZvciB0aGlzIHJ1bi4gIgorICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiT2ZmIGJ5IGRlZmF1bHQsIG1hdGNoaW5nIGNvbmZpZ3MvdHJhaW4uanNvbidzIGVuYWJsZWQ9ZmFsc2UsICIKKyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNvIGEgcGxhaW4gcnVuIGlzIHVuYWZmZWN0ZWQgdW5sZXNzIHRoaXMgaXMgcGFzc2VkIGV4cGxpY2l0bHkiKQorICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmFyZS10ZXJtLWZvcm11bGEiLCBkZWZhdWx0PU5vbmUsCisgICAgICAgICAgICAgICAgICAgICAgICBjaG9pY2VzPVsiaGFwYXhfYmluYXJ5IiwgImludmVyc2Vfc3FydF9mcmVxIl0sCisgICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJvdmVycmlkZSByYXJlX3Rlcm1fd2VpZ2h0aW5nLmZvcm11bGEuIE9ubHkgcmVhZCB3aGVuICIKKyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIi0tcmFyZS10ZXJtLXdlaWdodGluZyBpcyBhbHNvIHBhc3NlZCAob3IgYWxyZWFkeSBlbmFibGVkICIKKyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImluIHRoZSBjb25maWcpIikKKyAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJhcmUtdGVybS1oYXBheC13ZWlnaHQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PU5vbmUsCisgICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJvdmVycmlkZSByYXJlX3Rlcm1fd2VpZ2h0aW5nLmhhcGF4X3dlaWdodC4gT25seSByZWFkIGJ5ICIKKyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRoZSBoYXBheF9iaW5hcnkgZm9ybXVsYSIpCiAgICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKIAogICAgIGNmZyA9IGxvYWRfdHJhaW5fY29uZmlnKCkKQEAgLTE5OCw2ICsyMTAsMTIgQEAgZGVmIG1haW4oKSAtPiBOb25lOgogICAgICAgICBvdmVycmlkZXNbIm51bV9lcG9jaHMiXSA9IGFyZ3MuZXBvY2hzCiAgICAgaWYgYXJncy5sciBpcyBub3QgTm9uZToKICAgICAgICAgb3ZlcnJpZGVzWyJsZWFybmluZ19yYXRlIl0gPSBhcmdzLmxyCisgICAgaWYgYXJncy5yYXJlX3Rlcm1fd2VpZ2h0aW5nOgorICAgICAgICBvdmVycmlkZXNbInJhcmVfdGVybV93ZWlnaHRpbmdfZW5hYmxlZCJdID0gVHJ1ZQorICAgIGlmIGFyZ3MucmFyZV90ZXJtX2Zvcm11bGEgaXMgbm90IE5vbmU6CisgICAgICAgIG92ZXJyaWRlc1sicmFyZV90ZXJtX2Zvcm11bGEiXSA9IGFyZ3MucmFyZV90ZXJtX2Zvcm11bGEKKyAgICBpZiBhcmdzLnJhcmVfdGVybV9oYXBheF93ZWlnaHQgaXMgbm90IE5vbmU6CisgICAgICAgIG92ZXJyaWRlc1sicmFyZV90ZXJtX2hhcGF4X3dlaWdodCJdID0gYXJncy5yYXJlX3Rlcm1faGFwYXhfd2VpZ2h0CiAgICAgaWYgYXJncy53ZWlnaHRfZGVjYXkgaXMgbm90IE5vbmU6CiAgICAgICAgIG92ZXJyaWRlc1sid2VpZ2h0X2RlY2F5Il0gPSBhcmdzLndlaWdodF9kZWNheQogICAgIGlmIG92ZXJyaWRlczoKZGlmZiAtLWdpdCBhL3NyYy9tb2RlbHMvdHJhaW5fbG9vcC5weSBiL3NyYy9tb2RlbHMvdHJhaW5fbG9vcC5weQppbmRleCAwMmRkMzFiLi5jYWRhMDEyIDEwMDY0NAotLS0gYS9zcmMvbW9kZWxzL3RyYWluX2xvb3AucHkKKysrIGIvc3JjL21vZGVscy90cmFpbl9sb29wLnB5CkBAIC04OSwxNCArODksMjUgQEAgZGVmIHRyYWluX3N0ZXAoCiAgICAgYmF0Y2gucG9wKCJleGFtcGxlX2luZGV4IikgCiAgICAgI3dlIG5lZWQgdG8gbWFrZSBkYXRhIGFuZCBtb2RlbCBvbiBzYW1lIGRldmljZSAKICAgICBiYXRjaCA9IHtrIDogdC50byhkZXZpY2UpIGZvciBrICwgdCBpbiBiYXRjaC5pdGVtcygpIH0jbm93IGVhY2ggdGVuc29yIGluIGRpY3Qgb24gc2FtZSBkZXZpY2UgYXMgbW9kZWwgCi0gICAgb3V0cHV0ID0gbW9kZWwoKipiYXRjaCkgCi0gICAgbG9zcyA9IG91dHB1dC5sb3NzICAvIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyAKKworICAgIHdlaWdodHMgPSBiYXRjaC5wb3AoIndlaWdodHMiKQorICAgIG91dHB1dCA9IG1vZGVsKCoqYmF0Y2gpCisgICAgcGVyX3Bvc2l0aW9uX2xvc3MgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLmNyb3NzX2VudHJvcHkoCisgICAgICAgIG91dHB1dC5sb2dpdHMudmlldygtMSwgb3V0cHV0LmxvZ2l0cy5zaXplKC0xKSksCisgICAgICAgIGJhdGNoWyJsYWJlbHMiXS52aWV3KC0xKSwKKyAgICAgICAgaWdub3JlX2luZGV4PS0xMDAsCisgICAgICAgIHJlZHVjdGlvbj0ibm9uZSIsCisgICAgKQorICAgIG5fc2NvcmVkID0gKGJhdGNoWyJsYWJlbHMiXSAhPSAtMTAwKS5zdW0oKQorICAgIHdlaWdodGVkX2xvc3MgPSAocGVyX3Bvc2l0aW9uX2xvc3MgKiB3ZWlnaHRzLnZpZXcoLTEpKS5zdW0oKSAvIG5fc2NvcmVkCisgICAgbG9zcyA9IHdlaWdodGVkX2xvc3MgLyBncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMKKwogICAgIGxvc3MuYmFja3dhcmQoKQogICAgIAogICAgIGlmIGlzX3VwZGF0ZV9zdGVwIDogCiAgICAgICAgICB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obW9kZWwucGFyYW1ldGVycygpLCBtYXhfZ3JhZF9ub3JtKSAjIGRvIG5vcm1hbCBjbGlwaW5nIHNvIGdyYWRpZW50IHN0YXkgaW4gc2FtZSBkaXJlY3Rpb24gCiAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpOyBzY2hlZHVsZXIuc3RlcCgpOyBvcHRpbWl6ZXIuemVyb19ncmFkKCkKLSAgICByZXR1cm4gb3V0cHV0Lmxvc3MuaXRlbSgpIAorICAgIHJldHVybiB3ZWlnaHRlZF9sb3NzLml0ZW0oKSAKICAgICAKIAogCkBAIC0xNDAsNCArMTUxLDQgQEAgZGVmIGV2YWx1YXRlKAogICAgICAgICByZXN1bHRzW2YibGlzdF97a2V5fV9wIl0gPSBwcmVjaXNpb24KICAgICAgICAgcmVzdWx0c1tmImxpc3Rfe2tleX1fciJdID0gcmVjYWxsCiAgICAgICAgIHJlc3VsdHNbZiJsaXN0X3trZXl9X2YxIl0gPSBmMQotICAgIHJldHVybiByZXN1bHRzClwgTm8gbmV3bGluZSBhdCBlbmQgb2YgZmlsZQorICAgIHJldHVybiByZXN1bHRzCmRpZmYgLS1naXQgYS90ZXN0cy90ZXN0X3NjaGVkdWxlLnB5IGIvdGVzdHMvdGVzdF9zY2hlZHVsZS5weQppbmRleCBlNDM3MzcxLi42M2MyZDg3IDEwMDY0NAotLS0gYS90ZXN0cy90ZXN0X3NjaGVkdWxlLnB5CisrKyBiL3Rlc3RzL3Rlc3Rfc2NoZWR1bGUucHkKQEAgLTk2LDcgKzk2LDcgQEAgZGVmIHRlc3RfdHJhaW5fY29uZmlnX3JlYWRzX2V2ZXJ5X2tleV9pbl90aGVfZmlsZShjZmcpOgogICAgIHByb3NlIG5vdGVzIGFuZCBhcmUgZXhjbHVkZWQuCiAgICAgIiIiCiAgICAgc3RhdGVkID0ge2sgZm9yIGsgaW4gY2ZnLnJhdyBpZiBub3Qgay5zdGFydHN3aXRoKCJfIil9Ci0gICAgbmVzdGVkID0geyJzaG9ydF9zZW50ZW5jZV9maWx0ZXIifSAgICAgICAgICAjIGZsYXR0ZW5lZCBpbnRvIGZpbHRlcl8qIGZpZWxkcworICAgIG5lc3RlZCA9IHsic2hvcnRfc2VudGVuY2VfZmlsdGVyIiwgInJhcmVfdGVybV93ZWlnaHRpbmcifSAgIyBmbGF0dGVuZWQgaW50byBmaWVsZHMKICAgICBmaWVsZHMgPSB7Zi5uYW1lIGZvciBmIGluIGRhdGFjbGFzc2VzLmZpZWxkcyhjZmcpfQogICAgIG1pc3NpbmcgPSAoc3RhdGVkIC0gbmVzdGVkKSAtIGZpZWxkcwogICAgIGFzc2VydCBub3QgbWlzc2luZywgZiJjb25maWdzL3RyYWluLmpzb24gc3RhdGVzIHtzb3J0ZWQobWlzc2luZyl9LCBUcmFpbkNvbmZpZyBpZ25vcmVzIHRoZW0iCg=="
(REPO / "rare_term_weighting.patch").write_bytes(base64.b64decode(PATCH_B64))
run("git", "apply", "--check", "rare_term_weighting.patch", cwd=REPO)  # fail loudly, not silently
run("git", "apply", "rare_term_weighting.patch", cwd=REPO)

RARE_TERMS_B64 = "IiIiUmFyZS10ZXJtIHdlaWdodGluZzogdGVybSBmcmVxdWVuY3kgb3ZlciB0aGUgdHJhaW5pbmcgZG9tYWlucyBvbmx5LCBhbmQgdGhlCndlaWdodCBmb3JtdWxhcyB0aGF0IHR1cm4gdGhhdCBmcmVxdWVuY3kgaW50byBhIHBlci10ZXJtIGxvc3Mgd2VpZ2h0LgoKRnJlcXVlbmN5IGlzIGNvdW50ZWQgZnJvbSB3aGF0ZXZlciBgYGNvbmZpZ3MvZGF0YS5qc29uYGAgbmFtZXMgYXMKYGB0cmFpbl9kb21haW5zYGAgKGN1cnJlbnRseSBgYGNvcnBgYCArIGBgd2luZGBgKSwgcmVhZCBmcm9tIGBgRGF0YUNvbmZpZ2BgCnJhdGhlciB0aGFuIGhhcmRjb2RlZCwgc28gYSBmdXR1cmUgY2hhbmdlIHRvIHRoZSBzcGxpdCBpcyBwaWNrZWQgdXAgaGVyZSB0b28uCmBgZXF1aWBgIChkZXYpIGFuZCBgYGh0ZmxgYCAodGVzdCkgYXJlIG5ldmVyIGxvYWRlZCBieSB0aGlzIG1vZHVsZSAtLSBjb3VudGluZwplaXRoZXIgd291bGQgbGVhayBldmFsdWF0aW9uLWRvbWFpbiBpbmZvcm1hdGlvbiBpbnRvIGEgdHJhaW5pbmctdGltZSBzaWduYWwuCgpOZXcgZmlsZSwgbm90IGEgbW9kaWZpY2F0aW9uIG9mIGBgc3JjL2RhdGEvYWxpZ24ucHlgYDogQ0xBVURFLm1kIHJlc2VydmVzIHRoYXQKbW9kdWxlIGZvciBgYGFsaWduX2xhYmVsc2BgIC8gYGByZWNvdmVyX3Rva2VuX2xhYmVsc2BgIC8gYGBwb3NpdGl2ZV9yYXRlYGAuCmBgYWxpZ25fd2VpZ2h0c2BgIGJlbG93IG1pcnJvcnMgYGBhbGlnbl9sYWJlbHNgYCBpbiBsb2dpYyBvbmx5LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmZyb20gc3JjLmV2YWwuc3BhbnMgaW1wb3J0IGRlY29kZQpmcm9tIHNyYy5zdGF0cy5sb2FkaW5nIGltcG9ydCBEYXRhQ29uZmlnLCBEb2N1bWVudCwgbG9hZF9jb25maWcsIGxvYWRfZG9tYWluCgpJR05PUkVfV0VJR0hUID0gMC4wICAgICMgc3Vid29yZCBwb3NpdGlvbnMgd2l0aCBubyBmaXJzdC1zdWJ3b3JkIHdlaWdodCBvZiB0aGVpciBvd24KQkFTRUxJTkVfV0VJR0hUID0gMS4wICAjICJPIiB0b2tlbnMsIGFuZCBhbnkgdGVybSB0aGUgZnJlcXVlbmN5IHRhYmxlIGRvZXNuJ3QgY292ZXIKCgpkZWYgX3Rlcm1fc3RyaW5nKHRva2VuczogbGlzdFtzdHJdLCBzdGFydDogaW50LCBlbmQ6IGludCkgLT4gc3RyOgogICAgIiIiU2FtZSBjb25zdHJ1Y3Rpb24gYXMgYGBzcmMuZXZhbC5zdXJmYWNlLnNwYW5zX3RvX3VuaXF1ZV9saXN0YGAsIHNvIGEKICAgIHRlcm0gY291bnRlZCBoZXJlIGlzIHRoZSBpZGVudGljYWwgc3RyaW5nIHRvIHRoZSBvbmUgc2NvcmVkIHRoZXJlLiIiIgogICAgcmV0dXJuICIgIi5qb2luKHRva2Vuc1tzdGFydDplbmRdKS5sb3dlcigpCgoKZGVmIGNvdW50X3Rlcm1fZnJlcXVlbmNpZXMoCiAgICB0cmFpbl9kb21haW5zOiB0dXBsZVtzdHIsIC4uLl0sCiAgICAqLAogICAgZmlsdGVyX21heF90b2tlbnM6IGRpY3Rbc3RyLCBpbnQgfCBOb25lXSwKICAgIGRhdGFfY2ZnOiBEYXRhQ29uZmlnIHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdFtzdHIsIGludF06CiAgICAiIiJUZXJtIC0+IG9jY3VycmVuY2UgY291bnQsIG92ZXIgZXhhY3RseSB0aGUgc2VudGVuY2VzIGBgYnVpbGRfZXhhbXBsZXNgYAogICAgd2lsbCB0cmFpbiBvbi4KCiAgICBgYGZpbHRlcl9tYXhfdG9rZW5zYGAgbWFwcyBkb21haW4gLT4gdGhlIHNhbWUgYGA8PSBOIHRva2Vuc2BgIHRocmVzaG9sZAogICAgYGBUcmFpbkNvbmZpZy5maWx0ZXJfZm9yKGRvbWFpbiwgInRyYWluIilgYCByZXR1cm5zIGZvciB0aGF0IGRvbWFpbgogICAgKGBgTm9uZWBgIGZvciBubyBmaWx0ZXJpbmcpLCBzbyBhIHNlbnRlbmNlIHRoZSB0cmFpbmluZyBsb2FkZXIgZHJvcHMgaXMKICAgIGRyb3BwZWQgZnJvbSB0aGUgY291bnQgdG9vIC0tIGNvdW50aW5nIGl0IHdvdWxkIGluZmxhdGUgZnJlcXVlbmNpZXMgZm9yCiAgICB0ZXJtcyB0aGUgbW9kZWwgbmV2ZXIgYWN0dWFsbHkgc2Vlcy4KICAgICIiIgogICAgZGF0YV9jZmcgPSBkYXRhX2NmZyBvciBsb2FkX2NvbmZpZygpCiAgICBjb3VudHM6IENvdW50ZXJbc3RyXSA9IENvdW50ZXIoKQoKICAgIGZvciBkb21haW4gaW4gdHJhaW5fZG9tYWluczoKICAgICAgICB0aHJlc2hvbGQgPSBmaWx0ZXJfbWF4X3Rva2Vucy5nZXQoZG9tYWluKQogICAgICAgIGRvY3VtZW50czogbGlzdFtEb2N1bWVudF0gPSBsb2FkX2RvbWFpbihkb21haW4sIGRhdGFfY2ZnKQogICAgICAgIGZvciBkb2MgaW4gZG9jdW1lbnRzOgogICAgICAgICAgICBmb3IgdG9rZW5zLCBsYWJlbHMgaW4gZG9jLnNlbnRlbmNlczoKICAgICAgICAgICAgICAgIGlmIHRocmVzaG9sZCBpcyBub3QgTm9uZSBhbmQgbGVuKHRva2VucykgPD0gdGhyZXNob2xkOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBmb3Igc3RhcnQsIGVuZCBpbiBkZWNvZGUodG9rZW5zLCBsYWJlbHMsICJiaW8iKToKICAgICAgICAgICAgICAgICAgICBjb3VudHNbX3Rlcm1fc3RyaW5nKHRva2Vucywgc3RhcnQsIGVuZCldICs9IDEKCiAgICByZXR1cm4gZGljdChjb3VudHMpCgoKZGVmIHRlcm1fd2VpZ2h0KGZyZXE6IGludCwgKiwgZm9ybXVsYTogc3RyLCBoYXBheF93ZWlnaHQ6IGZsb2F0ID0gMi4wKSAtPiBmbG9hdDoKICAgICIiIk9uZSB0ZXJtJ3MgbG9zcyB3ZWlnaHQgZnJvbSBpdHMgdHJhaW5pbmctZG9tYWluIGZyZXF1ZW5jeS4KCiAgICBmb3JtdWxhOgogICAgICAiaGFwYXhfYmluYXJ5IiAgICAgIC0tIGhhcGF4X3dlaWdodCBpZiBmcmVxID09IDEsIGVsc2UgQkFTRUxJTkVfV0VJR0hUCiAgICAgICJpbnZlcnNlX3NxcnRfZnJlcSIgLS0gMS9zcXJ0KGZyZXEpOiBhIGhhcGF4IChmcmVxPTEpIHdlaWdocwogICAgICAgICAgICAgICAgICAgICAgICAgICAgIEJBU0VMSU5FX1dFSUdIVCBhbmQgZXZlcnkgb3RoZXIgdGVybSB3ZWlnaHMgbGVzcwogICAgIiIiCiAgICBpZiBmb3JtdWxhID09ICJoYXBheF9iaW5hcnkiOgogICAgICAgIHJldHVybiBoYXBheF93ZWlnaHQgaWYgZnJlcSA9PSAxIGVsc2UgQkFTRUxJTkVfV0VJR0hUCiAgICBpZiBmb3JtdWxhID09ICJpbnZlcnNlX3NxcnRfZnJlcSI6CiAgICAgICAgcmV0dXJuIEJBU0VMSU5FX1dFSUdIVCAvIG1hdGguc3FydChmcmVxKQogICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gZm9ybXVsYToge2Zvcm11bGEhcn0iKQoKCmRlZiB0b2tlbl93ZWlnaHRzX2Zvcl9zZW50ZW5jZSgKICAgIHRva2VuczogbGlzdFtzdHJdLAogICAgbGFiZWxzOiBsaXN0W3N0cl0sCiAgICB0ZXJtX2ZyZXF1ZW5jaWVzOiBkaWN0W3N0ciwgaW50XSwKICAgICosCiAgICBmb3JtdWxhOiBzdHIsCiAgICBoYXBheF93ZWlnaHQ6IGZsb2F0ID0gMi4wLAopIC0+IGxpc3RbZmxvYXRdOgogICAgIiIiT25lIHdlaWdodCBwZXIgREFUQVNFVCB0b2tlbiAocHJlLXN1YndvcmQpLCB2aWEgdGhlIHNhbWUgYGBkZWNvZGUoKWBgCiAgICBzcGFucyB1c2VkIGV2ZXJ5d2hlcmUgZWxzZS4gRXZlcnkgdG9rZW4gaW5zaWRlIG9uZSBzcGFuIHNoYXJlcyB0aGF0CiAgICBzcGFuJ3Mgd2VpZ2h0OyAiTyIgdG9rZW5zIGdldCBgYEJBU0VMSU5FX1dFSUdIVGBgLiBBIHRlcm0gbWlzc2luZyBmcm9tCiAgICBgYHRlcm1fZnJlcXVlbmNpZXNgYCAoc2hvdWxkIG5vdCBoYXBwZW4gZm9yIHRyYWluLWRvbWFpbiBzZW50ZW5jZXMsIGJ1dAogICAgZ3VhcmRzIGEgY2FsbGVyIHRoYXQgcGFzc2VzIGFuIGluY29tcGxldGUgdGFibGUpIGFsc28gZ2V0cwogICAgYGBCQVNFTElORV9XRUlHSFRgYCByYXRoZXIgdGhhbiByYWlzaW5nLgogICAgIiIiCiAgICB3ZWlnaHRzID0gW0JBU0VMSU5FX1dFSUdIVF0gKiBsZW4odG9rZW5zKQogICAgZm9yIHN0YXJ0LCBlbmQgaW4gZGVjb2RlKHRva2VucywgbGFiZWxzLCAiYmlvIik6CiAgICAgICAgZnJlcSA9IHRlcm1fZnJlcXVlbmNpZXMuZ2V0KF90ZXJtX3N0cmluZyh0b2tlbnMsIHN0YXJ0LCBlbmQpKQogICAgICAgIHcgPSAodGVybV93ZWlnaHQoZnJlcSwgZm9ybXVsYT1mb3JtdWxhLCBoYXBheF93ZWlnaHQ9aGFwYXhfd2VpZ2h0KQogICAgICAgICAgICAgaWYgZnJlcSBpcyBub3QgTm9uZSBlbHNlIEJBU0VMSU5FX1dFSUdIVCkKICAgICAgICBmb3IgaSBpbiByYW5nZShzdGFydCwgZW5kKToKICAgICAgICAgICAgd2VpZ2h0c1tpXSA9IHcKICAgIHJldHVybiB3ZWlnaHRzCgoKZGVmIGFsaWduX3dlaWdodHMod29yZF9pZHM6IGxpc3RbaW50IHwgTm9uZV0sIHRva2VuX3dlaWdodHM6IGxpc3RbZmxvYXRdKSAtPiBsaXN0W2Zsb2F0XToKICAgICIiIk1pcnJvcnMgYGBzcmMuZGF0YS5hbGlnbi5hbGlnbl9sYWJlbHNgYCBleGFjdGx5LCBmb3IgZmxvYXRzIGluc3RlYWQgb2YKICAgIGxhYmVsIGlkcy4gVGhlIGZpcnN0IHN1YndvcmQgb2YgZWFjaCBkYXRhc2V0IHRva2VuIGNhcnJpZXMgaXRzIHdlaWdodDsKICAgIGV2ZXJ5IGNvbnRpbnVhdGlvbiBzdWJ3b3JkIGFuZCBzcGVjaWFsIHRva2VuIGdldHMgYGBJR05PUkVfV0VJR0hUYGAgKDAuMCkKICAgIC0tIHRob3NlIHBvc2l0aW9ucyBhbHJlYWR5IGNhcnJ5IGBgbGFiZWxzID09IC0xMDBgYCwgc28gdGhpcyBpcyBhIGJlbHQtCiAgICBhbmQtc3VzcGVuZGVycyBzZW50aW5lbCwgbmV2ZXIgdGhlIHRoaW5nIHRoZSBtYXNrIGFjdHVhbGx5IHJlbGllcyBvbi4KICAgICIiIgogICAgcHJldjogaW50IHwgTm9uZSA9IE5vbmUKICAgIG91dDogbGlzdFtmbG9hdF0gPSBbXQogICAgZm9yIHdvcmRfaWQgaW4gd29yZF9pZHM6CiAgICAgICAgaWYgd29yZF9pZCBpcyBOb25lOgogICAgICAgICAgICBvdXQuYXBwZW5kKElHTk9SRV9XRUlHSFQpCiAgICAgICAgZWxpZiB3b3JkX2lkICE9IHByZXY6CiAgICAgICAgICAgIG91dC5hcHBlbmQodG9rZW5fd2VpZ2h0c1t3b3JkX2lkXSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBvdXQuYXBwZW5kKElHTk9SRV9XRUlHSFQpCiAgICAgICAgcHJldiA9IHdvcmRfaWQKICAgIHJldHVybiBvdXQK"
(REPO / "src/data/rare_terms.py").write_bytes(base64.b64decode(RARE_TERMS_B64))

TEST_RARE_TERMS_B64 = "IiIiVGVzdHMgZm9yIHNyYy5kYXRhLnJhcmVfdGVybXMuCgpEZWxpYmVyYXRlbHkgaW5kZXBlbmRlbnQgb2YgYW55IHRva2VuaXplcjogYWxpZ25fd2VpZ2h0cyBvcGVyYXRlcyBvbgp3b3JkX2lkcyBsaXN0cyB0aGUgc2FtZSB3YXkgYWxpZ25fbGFiZWxzIGRvZXMsIHNvIGl0IGlzIHRlc3RhYmxlIHdpdGgKaGFuZC1jcmFmdGVkIHdvcmRfaWRzIGV4YWN0bHkgbGlrZSB0ZXN0X2FsaWdubWVudF9nYXRlLnB5J3MgaGFuZC1jaGVja2VkCnNlbnRlbmNlIC0tIG5vIG5ldHdvcmssIG5vIG1vZGVsIGRvd25sb2FkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCgppbXBvcnQgcHl0ZXN0Cgpmcm9tIHNyYy5kYXRhLnJhcmVfdGVybXMgaW1wb3J0ICgKICAgIEJBU0VMSU5FX1dFSUdIVCwKICAgIGFsaWduX3dlaWdodHMsCiAgICBjb3VudF90ZXJtX2ZyZXF1ZW5jaWVzLAogICAgdGVybV93ZWlnaHQsCiAgICB0b2tlbl93ZWlnaHRzX2Zvcl9zZW50ZW5jZSwKKQpmcm9tIHNyYy5zdGF0cy5sb2FkaW5nIGltcG9ydCBsb2FkX2NvbmZpZwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBTY29wZTogY29ycCArIHdpbmQgb25seSwgbmV2ZXIgZXF1aSBvciBodGZsCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKZGVmIHRlc3RfZnJlcXVlbmN5X3Njb3BlX21hdGNoZXNfY29uZmlndXJlZF90cmFpbl9kb21haW5zKCk6CiAgICAiIiJXaGF0ZXZlciBjb25maWdzL2RhdGEuanNvbiBjYWxscyB0cmFpbl9kb21haW5zIGlzIHdoYXQgZ2V0cyBjb3VudGVkIC0tCiAgICBub3QgYSBoYXJkY29kZWQgbGlzdCBoZXJlLiIiIgogICAgZGF0YV9jZmcgPSBsb2FkX2NvbmZpZygpCiAgICBhc3NlcnQgZGF0YV9jZmcudHJhaW5fZG9tYWlucyA9PSAoImNvcnAiLCAid2luZCIpLCAoCiAgICAgICAgInRoaXMgdGVzdCdzIGFzc3VtcHRpb25zIChhbmQgdGhlIGxlYWthZ2UgY2hlY2tzIGJlbG93KSBhcmUgcGlubmVkIHRvICIKICAgICAgICAidGhlIGFkb3B0ZWQgc3BsaXQ7IHJlLWNoZWNrIHRoZW0gaWYgdHJhaW5fZG9tYWlucyBldmVyIGNoYW5nZXMiCiAgICApCgoKZGVmIHRlc3RfaHRmbF9vbmx5X3Rlcm1faXNfYWJzZW50KCk6CiAgICAiIiInaGVhcnQgZmFpbHVyZScgaXMgaHRmbCdzIG93biBleGFtcGxlIHRlcm0gKERhdGFfc3RhdHMubWQsIG5lc3RlZC10ZXJtCiAgICB0YWJsZSkgYW5kIGRvZXMgbm90IG9jY3VyIGluIGNvcnAgb3Igd2luZC4gSWYgaXQgZXZlciBzaG93ZWQgdXAgaGVyZSwgdGhlCiAgICBsb2FkZXIgcHVsbGVkIGluIGEgZG9tYWluIGl0IHNob3VsZCBub3QgaGF2ZS4iIiIKICAgIGRhdGFfY2ZnID0gbG9hZF9jb25maWcoKQogICAgZnJlcXMgPSBjb3VudF90ZXJtX2ZyZXF1ZW5jaWVzKAogICAgICAgIGRhdGFfY2ZnLnRyYWluX2RvbWFpbnMsCiAgICAgICAgZmlsdGVyX21heF90b2tlbnM9eyJjb3JwIjogTm9uZSwgIndpbmQiOiAyfSwKICAgICAgICBkYXRhX2NmZz1kYXRhX2NmZywKICAgICkKICAgIGFzc2VydCAiaGVhcnQgZmFpbHVyZSIgbm90IGluIGZyZXFzCgoKZGVmIHRlc3RfZXF1aV90ZXJtc19kb19ub3RfaW5mbGF0ZV9jb3VudHMoKToKICAgICIiIkNvdW50aW5nIGNvcnArd2luZCtlcXVpIG11c3QgZ2l2ZSBzdHJpY3RseSBtb3JlIG9jY3VycmVuY2VzIHRoYW4KICAgIGNvcnArd2luZCBhbG9uZSAtLSBwcm9vZiBlcXVpIHdhcyBuZXZlciB0b3VjaGVkIGJ5IHRoZSByZWFsIGNhbGwsIG5vdAogICAganVzdCBhbiBhYnNlbmNlLW9mLWVycm9yLiIiIgogICAgZGF0YV9jZmcgPSBsb2FkX2NvbmZpZygpCiAgICBjb3JyZWN0ID0gY291bnRfdGVybV9mcmVxdWVuY2llcygKICAgICAgICAoImNvcnAiLCAid2luZCIpLCBmaWx0ZXJfbWF4X3Rva2Vucz17ImNvcnAiOiBOb25lLCAid2luZCI6IDJ9LCBkYXRhX2NmZz1kYXRhX2NmZykKICAgIHdpdGhfZXF1aSA9IGNvdW50X3Rlcm1fZnJlcXVlbmNpZXMoCiAgICAgICAgKCJjb3JwIiwgImVxdWkiLCAid2luZCIpLCBmaWx0ZXJfbWF4X3Rva2Vucz17ImNvcnAiOiBOb25lLCAiZXF1aSI6IE5vbmUsICJ3aW5kIjogMn0sCiAgICAgICAgZGF0YV9jZmc9ZGF0YV9jZmcpCiAgICBhc3NlcnQgc3VtKHdpdGhfZXF1aS52YWx1ZXMoKSkgPiBzdW0oY29ycmVjdC52YWx1ZXMoKSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgV2VpZ2h0IGZvcm11bGFzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKZGVmIHRlc3RfaGFwYXhfYmluYXJ5X2Zvcm11bGEoKToKICAgIGFzc2VydCB0ZXJtX3dlaWdodCgxLCBmb3JtdWxhPSJoYXBheF9iaW5hcnkiLCBoYXBheF93ZWlnaHQ9Mi4wKSA9PSAyLjAKICAgIGFzc2VydCB0ZXJtX3dlaWdodCgyLCBmb3JtdWxhPSJoYXBheF9iaW5hcnkiLCBoYXBheF93ZWlnaHQ9Mi4wKSA9PSBCQVNFTElORV9XRUlHSFQKICAgIGFzc2VydCB0ZXJtX3dlaWdodCgzNTAsIGZvcm11bGE9ImhhcGF4X2JpbmFyeSIsIGhhcGF4X3dlaWdodD0yLjApID09IEJBU0VMSU5FX1dFSUdIVAoKCmRlZiB0ZXN0X2ludmVyc2Vfc3FydF9mcmVxX2Zvcm11bGEoKToKICAgIGFzc2VydCB0ZXJtX3dlaWdodCgxLCBmb3JtdWxhPSJpbnZlcnNlX3NxcnRfZnJlcSIpID09IHB5dGVzdC5hcHByb3goMS4wKQogICAgYXNzZXJ0IHRlcm1fd2VpZ2h0KDQsIGZvcm11bGE9ImludmVyc2Vfc3FydF9mcmVxIikgPT0gcHl0ZXN0LmFwcHJveCgwLjUpCiAgICBhc3NlcnQgdGVybV93ZWlnaHQoMzUwLCBmb3JtdWxhPSJpbnZlcnNlX3NxcnRfZnJlcSIpID09IHB5dGVzdC5hcHByb3goMSAvIG1hdGguc3FydCgzNTApKQoKCmRlZiB0ZXN0X3Vua25vd25fZm9ybXVsYV9yZWplY3RlZCgpOgogICAgd2l0aCBweXRlc3QucmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgIHRlcm1fd2VpZ2h0KDEsIGZvcm11bGE9Im5vdF9hX3JlYWxfZm9ybXVsYSIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFRva2VuLWxldmVsIHdlaWdodHMgZnJvbSBvbmUgc2VudGVuY2UKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpkZWYgdGVzdF9hbGxfdG9rZW5zX2luX29uZV9zcGFuX3NoYXJlX3RoZV9zYW1lX3dlaWdodCgpOgogICAgdG9rZW5zID0gWyJteW9jeXRlIiwgImh5cGVydHJvcGh5IiwgIndhcyIsICJvYnNlcnZlZCJdCiAgICBsYWJlbHMgPSBbIkIiLCAiSSIsICJPIiwgIk8iXQogICAgZnJlcXMgPSB7Im15b2N5dGUgaHlwZXJ0cm9waHkiOiAxfQogICAgd2VpZ2h0cyA9IHRva2VuX3dlaWdodHNfZm9yX3NlbnRlbmNlKAogICAgICAgIHRva2VucywgbGFiZWxzLCBmcmVxcywgZm9ybXVsYT0iaGFwYXhfYmluYXJ5IiwgaGFwYXhfd2VpZ2h0PTMuMCkKICAgIGFzc2VydCB3ZWlnaHRzID09IFszLjAsIDMuMCwgQkFTRUxJTkVfV0VJR0hULCBCQVNFTElORV9XRUlHSFRdCgoKZGVmIHRlc3RfdGVybV9taXNzaW5nX2Zyb21fdGFibGVfZmFsbHNfYmFja190b19iYXNlbGluZSgpOgogICAgdG9rZW5zID0gWyJub3ZlbCIsICJ0ZXJtIl0KICAgIGxhYmVscyA9IFsiQiIsICJJIl0KICAgIHdlaWdodHMgPSB0b2tlbl93ZWlnaHRzX2Zvcl9zZW50ZW5jZSh0b2tlbnMsIGxhYmVscywge30sIGZvcm11bGE9ImludmVyc2Vfc3FydF9mcmVxIikKICAgIGFzc2VydCB3ZWlnaHRzID09IFtCQVNFTElORV9XRUlHSFQsIEJBU0VMSU5FX1dFSUdIVF0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgU3Vid29yZCBhbGlnbm1lbnQgLS0gaGFuZC1jcmFmdGVkIHdvcmRfaWRzLCBzYW1lIHN0eWxlIGFzCiMgdGVzdF9hbGlnbm1lbnRfZ2F0ZS5weTo6dGVzdF9hbGlnbl9sYWJlbHNfb25fYV9oYW5kX2NoZWNrZWRfc2VudGVuY2UKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpkZWYgdGVzdF9hbGlnbl93ZWlnaHRzX29uX2FfaGFuZF9jaGVja2VkX3NlbnRlbmNlKCk6CiAgICAiIiInc2VsZi1lbXBsb3llZCcgdG9rZW5pemVzIHRvIHNldmVyYWwgd29yZHBpZWNlczsgd29yZF9pZHMgYmVsb3cgaXMgYQogICAgcGxhdXNpYmxlIGZhc3QtdG9rZW5pemVyIG91dHB1dCBmb3IKICAgIFsiVGhlIiwgInNlbGYtZW1wbG95ZWQiLCAibm9uIiwgIi0iLCAiZ292ZXJubWVudGFsIiwgImJvZHkiLCAiLiJdCiAgICB3aGVyZSAnc2VsZi1lbXBsb3llZCcgc3BsaXRzIGludG8gMyBwaWVjZXMgYW5kIGV2ZXJ5dGhpbmcgZWxzZSBpbnRvIDEsCiAgICBwbHVzIGEgW0NMU10vW1NFUF0gc3BlY2lhbCBhdCBlYWNoIGVuZCAod29yZF9pZCBOb25lKS4iIiIKICAgIHRva2VuX3dlaWdodHMgPSBbMS4wLCA1LjAsIDEuMCwgMS4wLCAxLjAsIDEuMCwgMS4wXSAgIyBvbmUgd2VpZ2h0IHBlciBkYXRhc2V0IHRva2VuCiAgICB3b3JkX2lkcyA9IFtOb25lLCAwLCAxLCAxLCAxLCAyLCAzLCA0LCA1LCA2LCBOb25lXQoKICAgIGFsaWduZWQgPSBhbGlnbl93ZWlnaHRzKHdvcmRfaWRzLCB0b2tlbl93ZWlnaHRzKQoKICAgIGFzc2VydCBsZW4oYWxpZ25lZCkgPT0gbGVuKHdvcmRfaWRzKQogICAgZmlyc3RfcG9zaXRpb25zID0gW2kgZm9yIGksIHcgaW4gZW51bWVyYXRlKHdvcmRfaWRzKQogICAgICAgICAgICAgICAgICAgICAgIGlmIHcgaXMgbm90IE5vbmUgYW5kIChpID09IDAgb3Igd29yZF9pZHNbaSAtIDFdICE9IHcpXQogICAgYXNzZXJ0IGxlbihmaXJzdF9wb3NpdGlvbnMpID09IGxlbih0b2tlbl93ZWlnaHRzKSwgIm9uZSB3ZWlnaHQgcGVyIGRhdGFzZXQgdG9rZW4iCiAgICBhc3NlcnQgW2FsaWduZWRbaV0gZm9yIGkgaW4gZmlyc3RfcG9zaXRpb25zXSA9PSB0b2tlbl93ZWlnaHRzCiAgICBhc3NlcnQgYWxsKGFsaWduZWRbaV0gPT0gMC4wIGZvciBpIGluIHJhbmdlKGxlbih3b3JkX2lkcykpIGlmIGkgbm90IGluIGZpcnN0X3Bvc2l0aW9ucykKCgpkZWYgdGVzdF9hbGlnbl93ZWlnaHRzX21hdGNoZXNfYWxpZ25fbGFiZWxzX2ZpcnN0X3N1YndvcmRfcG9zaXRpb25zKCk6CiAgICAiIiJhbGlnbl93ZWlnaHRzIGFuZCBhbGlnbl9sYWJlbHMgbXVzdCBhZ3JlZSBvbiBXSElDSCBwb3NpdGlvbiBpcyBhCiAgICB0b2tlbidzIGZpcnN0IHN1YndvcmQgLS0gdGhpcyBpcyB0aGUgcHJvcGVydHkgdGhlIFQ2IGdhdGUgY2FyZXMgYWJvdXQsCiAgICBub3cgY2hlY2tlZCBmb3Igd2VpZ2h0cyB0b28uIiIiCiAgICBmcm9tIHNyYy5kYXRhLmFsaWduIGltcG9ydCBMQUJFTDJJRCwgYWxpZ25fbGFiZWxzCgogICAgd29yZF9pZHMgPSBbTm9uZSwgMCwgMSwgMSwgMSwgMiwgMywgNCwgNSwgNiwgTm9uZV0KICAgIGxhYmVscyA9IFsiTyIsICJCIiwgIkIiLCAiSSIsICJJIiwgIkkiLCAiTyJdCiAgICB0b2tlbl93ZWlnaHRzID0gWzEuMCwgMi4wLCAyLjAsIDIuMCwgMi4wLCAyLjAsIDEuMF0KCiAgICBhbGlnbmVkX2xhYmVscyA9IGFsaWduX2xhYmVscyh3b3JkX2lkcywgbGFiZWxzLCBMQUJFTDJJRCkKICAgIGFsaWduZWRfd2VpZ2h0cyA9IGFsaWduX3dlaWdodHMod29yZF9pZHMsIHRva2VuX3dlaWdodHMpCgogICAgaWdub3JlZF9ieV9sYWJlbHMgPSBbaSBmb3IgaSwgdiBpbiBlbnVtZXJhdGUoYWxpZ25lZF9sYWJlbHMpIGlmIHYgPT0gLTEwMF0KICAgIGlnbm9yZWRfYnlfd2VpZ2h0cyA9IFtpIGZvciBpLCB2IGluIGVudW1lcmF0ZShhbGlnbmVkX3dlaWdodHMpIGlmIHYgPT0gMC4wXQogICAgYXNzZXJ0IGlnbm9yZWRfYnlfbGFiZWxzID09IGlnbm9yZWRfYnlfd2VpZ2h0cwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKIyBSZWdyZXNzaW9uIGd1YXJkIGZvciB0cmFpbl9zdGVwJ3Mgd2VpZ2h0ZWQtbG9zcyBtYXRoICh0cmFpbl9sb29wLnB5IGlzCiMgaGFuZC13cml0dGVuIGFuZCBvZmYtbGltaXRzIGhlcmUgLS0gdGhpcyB0ZXN0cyB0aGUgTUFUSCB0aGUgcGF0Y2ggYXBwbGllcywKIyBub3QgdHJhaW5fc3RlcCBpdHNlbGYpLiBOZWVkcyB0b3JjaDsgc2tpcHBlZCB3aGVyZSBpdCBpc24ndCBpbnN0YWxsZWQuCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKZGVmIHRlc3Rfd2VpZ2h0ZWRfbG9zc193aXRoX2FsbF9vbmVzX2VxdWFsc19wbGFpbl9tZWFuX2Nyb3NzX2VudHJvcHkoKToKICAgICIiIlRoZSBvbmUgcHJvcGVydHkgdGhhdCBtdXN0IGhvbGQgYmVmb3JlIHRydXN0aW5nIEFOWSBydW46IHdpdGggZXZlcnkKICAgIHdlaWdodCA9PSAxLjAgKHJhcmVfdGVybV93ZWlnaHRpbmcuZW5hYmxlZD1mYWxzZSwgb3IgZXZlcnkgcG9zaXRpb24KICAgIGJhc2VsaW5lKSwgdHJhaW5fc3RlcCdzIHdlaWdodGVkIGxvc3MgbXVzdCBlcXVhbCBwbGFpbiBtZWFuCiAgICBjcm9zcy1lbnRyb3B5IGV4YWN0bHkgLS0gc2FtZSBmb3JtdWxhIHRyYWluX2xvb3AucHkncyBwYXRjaCB1c2VzLAogICAgY2hlY2tlZCBoZXJlIGluZGVwZW5kZW50bHkgb2YgdGhhdCBwcm90ZWN0ZWQgZmlsZS4iIiIKICAgIHRvcmNoID0gcHl0ZXN0LmltcG9ydG9yc2tpcCgidG9yY2giKQogICAgaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDApCiAgICBiYXRjaF9zaXplLCBzZXFfbGVuLCBudW1fbGFiZWxzID0gNCwgMTIsIDMKICAgIGxvZ2l0cyA9IHRvcmNoLnJhbmRuKGJhdGNoX3NpemUsIHNlcV9sZW4sIG51bV9sYWJlbHMpCiAgICBsYWJlbHMgPSB0b3JjaC5yYW5kaW50KDAsIG51bV9sYWJlbHMsIChiYXRjaF9zaXplLCBzZXFfbGVuKSkKICAgICMgc3ByaW5rbGUgaW4gc29tZSBpZ25vcmVkICgtMTAwKSBwb3NpdGlvbnMsIGxpa2UgcmVhbCBwYWRkaW5nL2NvbnRpbnVhdGlvbiBzdWJ3b3JkcwogICAgbGFiZWxzWzAsIC0zOl0gPSAtMTAwCiAgICBsYWJlbHNbMiwgOjJdID0gLTEwMAogICAgd2VpZ2h0cyA9IHRvcmNoLm9uZXMoYmF0Y2hfc2l6ZSwgc2VxX2xlbikKCiAgICBleHBlY3RlZCA9IEYuY3Jvc3NfZW50cm9weSgKICAgICAgICBsb2dpdHMudmlldygtMSwgbnVtX2xhYmVscyksIGxhYmVscy52aWV3KC0xKSwgaWdub3JlX2luZGV4PS0xMDAsIHJlZHVjdGlvbj0ibWVhbiIpCgogICAgcGVyX3Bvc2l0aW9uX2xvc3MgPSBGLmNyb3NzX2VudHJvcHkoCiAgICAgICAgbG9naXRzLnZpZXcoLTEsIG51bV9sYWJlbHMpLCBsYWJlbHMudmlldygtMSksIGlnbm9yZV9pbmRleD0tMTAwLCByZWR1Y3Rpb249Im5vbmUiKQogICAgbl9zY29yZWQgPSAobGFiZWxzICE9IC0xMDApLnN1bSgpCiAgICB3ZWlnaHRlZF9sb3NzID0gKHBlcl9wb3NpdGlvbl9sb3NzICogd2VpZ2h0cy52aWV3KC0xKSkuc3VtKCkgLyBuX3Njb3JlZAoKICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZSh3ZWlnaHRlZF9sb3NzLCBleHBlY3RlZCwgYXRvbD0xZS02KQo="
(REPO / "tests/test_rare_terms.py").write_bytes(base64.b64decode(TEST_RARE_TERMS_B64))

print(subprocess.run(["git", "status", "--short"], cwd=REPO,
                     capture_output=True, text=True).stdout)


In [ ]:
# 4. Pinned installs. torch/numpy are Kaggle's -- forcing them breaks its CUDA
#    build, exactly as on Colab.
run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.16.1', 'tokenizers==0.23.1', 'safetensors==0.8.0',
    'huggingface_hub==1.29.0', 'sentencepiece==0.2.2', 'protobuf==7.36.0',
    'PyYAML==6.0.3', 'pytest==8.3.2')

def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0
HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('seqeval:', 'installed' if HAVE_SEQEVAL else 'UNAVAILABLE -- training unaffected')


In [ ]:
# 5. Run the full test suite -- including tests/test_rare_terms.py, which now has
#    torch available, so the weighted-loss-math regression test actually runs here
#    (it only skips locally, where torch is missing).
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
run(sys.executable, '-m', 'pytest', 'tests/', '-v', *skip, cwd=REPO)


In [ ]:
# 6. The three conditions: baseline (weighting off), hapax_binary, inverse_sqrt_freq.
#    5 seeds each -- 15 runs total, matching the project's own seed convention (T8/T9).
#    Model/LR/epochs = T10's selected config. Swap to bert-base-cased/3e-5 here for a
#    faster, non-final run instead.
import json

MODEL, LR, EPOCHS = 'microsoft/deberta-v3-base', '1e-5', 5
SEEDS = (42, 43, 44, 45, 46)
CONDITIONS = [
    ('baseline', []),
    ('hapax_binary', ['--rare-term-weighting', '--rare-term-formula', 'hapax_binary']),
    ('inverse_sqrt_freq', ['--rare-term-weighting', '--rare-term-formula', 'inverse_sqrt_freq']),
]

for name, flags in CONDITIONS:
    group = f'rare_term_weighting/{name}'
    src = REPO / 'results/runs' / group
    print(f'\n########## {name} ##########')
    for s in SEEDS:
        run(sys.executable, '-m', 'src.models.run_train',
            '--model', MODEL, '--lr', LR, '--epochs', EPOCHS,
            '--group', group, '--seed', s,
            '--reason', f'rare-term weighting: {name}', *flags, cwd=REPO)
        r = json.loads((src / f'seed_{s}.json').read_text())
        print(f"    SEED {s}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
              f"htfl={r['htfl_f1']:.4f} collapsed={r['collapsed']} {r['wall_time_sec']}s")
    dest = WORK / group
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest)
    print(f'  -> copied to {dest} (persists as notebook Output; survives a dropped session)')


In [ ]:
# 7. What to download: everything under /kaggle/working/rare_term_weighting.
#    Drop these folders into results/runs/rare_term_weighting/ locally (on the
#    feature/rare-term-weighting branch), then aggregate/compare the 3 conditions.
#    T11 (hapax-vs-rest breakdown on htfl) still needs to be built before these numbers
#    answer "did it work on hapax specifically" -- this cell only gets the runs done.
for p in sorted((WORK / 'rare_term_weighting').rglob('seed_*.json')):
    print(' ', p.relative_to(WORK), f'{p.stat().st_size/1024:.0f} KB')
